# Week 1 — NLP Preprocessing of Task Descriptions

**Project:** AI-Powered Task Management System
**Input:** `data/processed/tasks_clean.csv` (output of the EDA & cleaning notebook)
**Prepared by:** Vijayasiva

**Preprocessing pipeline (per the Week 1 plan)**
1. Lowercasing
2. Punctuation & special-character removal
3. Tokenization
4. Stop-word removal
5. Stemming vs. lemmatization (comparison, then lemmatization applied)
6. Verification that the processed text is clean and ready for Week 2 feature extraction (TF-IDF / embeddings)

In [1]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

for pkg in ["punkt_tab", "stopwords", "wordnet"]:
    nltk.download(pkg, quiet=True)

df = pd.read_csv("../data/processed/tasks_clean.csv")
print("Rows:", len(df))
df[["task_id", "task_description", "category"]].head()

Rows: 8000


,task_id,task_description,category
0,TASK-12157,Implement pagination and lazy loading for the ...,Feature
1,TASK-10453,Correct the wrong currency formatting shown on...,Bug Fix
2,TASK-12716,Add role-based access control to the database ...,Feature
3,TASK-16240,Implement pagination and lazy loading for the ...,Feature
4,TASK-15364,Write unit tests covering edge cases of the au...,Testing


## 1. Lowercasing and Punctuation / Special-Character Removal

In [2]:
def normalize_text(text: str) -> str:
    text = text.lower()                        # lowercase
    text = re.sub(r"[^a-z0-9\s]", " ", text)   # remove punctuation & special characters
    text = re.sub(r"\s+", " ", text).strip()   # collapse repeated whitespace
    return text

df["description_normalized"] = df["task_description"].apply(normalize_text)

# Urgency markers like "URGENT:" or "ASAP -" become plain words and casing noise disappears
df.loc[df["task_description"].str.contains("URGENT|ASAP", regex=True),
       ["task_description", "description_normalized"]].head(3)

,task_description,description_normalized
25,ASAP - Correct the wrong currency formatting s...,asap correct the wrong currency formatting sho...
33,ASAP - Fix the timeout issue in the inventory ...,asap fix the timeout issue in the inventory mo...
74,ASAP - Correct the wrong currency formatting s...,asap correct the wrong currency formatting sho...


## 2. Tokenization

In [3]:
df["tokens"] = df["description_normalized"].apply(word_tokenize)
df[["description_normalized", "tokens"]].head(3)

,description_normalized,tokens
0,implement pagination and lazy loading for the ...,"[implement, pagination, and, lazy, loading, fo..."
1,correct the wrong currency formatting shown on...,"[correct, the, wrong, currency, formatting, sh..."
2,add role based access control to the database ...,"[add, role, based, access, control, to, the, d..."


## 3. Stop-word Removal

In [4]:
stop_words = set(stopwords.words("english"))

df["tokens_no_stop"] = df["tokens"].apply(
    lambda toks: [t for t in toks if t not in stop_words])

before = df["tokens"].str.len().sum()
after = df["tokens_no_stop"].str.len().sum()
print(f"Tokens before stop-word removal: {before:,}")
print(f"Tokens after stop-word removal:  {after:,}  ({(before-after)/before:.1%} removed)")

Tokens before stop-word removal: 78,529
Tokens after stop-word removal:  53,698  (31.6% removed)


## 4. Stemming vs. Lemmatization

We compare both on sample words from the corpus, then apply **lemmatization** for the final pipeline — it returns real dictionary words (better for interpretable TF-IDF features in Week 2), whereas stemming can produce truncated non-words.

In [5]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

samples = ["fixes", "deployed", "testing", "dependencies", "credentials",
           "queries", "caching", "settings", "scenarios", "improving"]
comparison = pd.DataFrame({
    "word": samples,
    "Porter stem": [stemmer.stem(w) for w in samples],
    "WordNet lemma": [lemmatizer.lemmatize(w) for w in samples],
})
comparison

,word,Porter stem,WordNet lemma
0,fixes,fix,fix
1,deployed,deploy,deployed
2,testing,test,testing
3,dependencies,depend,dependency
4,credentials,credenti,credential
5,queries,queri,query
6,caching,cach,caching
7,settings,set,setting
8,scenarios,scenario,scenario
9,improving,improv,improving


In [6]:
df["tokens_final"] = df["tokens_no_stop"].apply(
    lambda toks: [lemmatizer.lemmatize(t) for t in toks])
df["description_processed"] = df["tokens_final"].str.join(" ")

df[["task_description", "description_processed"]].head(5)

,task_description,description_processed
0,Implement pagination and lazy loading for the ...,implement pagination lazy loading landing page
1,Correct the wrong currency formatting shown on...,correct wrong currency formatting shown login ...
2,Add role-based access control to the database ...,add role based access control database layer
3,Implement pagination and lazy loading for the ...,implement pagination lazy loading email scheduler
4,Write unit tests covering edge cases of the au...,write unit test covering edge case authenticat...


## 5. Verification — Is the Processed Text Clean?

In [7]:
all_tokens = [t for toks in df["tokens_final"] for t in toks]
vocab = set(all_tokens)

print("Total tokens:", f"{len(all_tokens):,}")
print("Vocabulary size:", len(vocab))
print("Empty processed descriptions:", (df["description_processed"].str.len() == 0).sum())
print("Tokens containing punctuation/special chars:",
      sum(1 for t in vocab if re.search(r"[^a-z0-9]", t)))
print("Uppercase characters remaining:",
      df["description_processed"].str.contains(r"[A-Z]").sum())
print("Stop words remaining:",
      sum(1 for t in all_tokens if t in stop_words))

Total tokens: 53,698
Vocabulary size: 207
Empty processed descriptions: 0
Tokens containing punctuation/special chars: 0
Uppercase characters remaining: 0
Stop words remaining: 0


In [8]:
from collections import Counter
top20 = pd.DataFrame(Counter(all_tokens).most_common(20), columns=["token", "count"])
top20

,token,count
0,fix,1404
1,page,1225
2,service,1185
3,add,1026
4,test,968
5,module,965
6,user,884
7,implement,746
8,authentication,613
9,create,565


**Verification result:** every description survived preprocessing (none became empty), the vocabulary contains no punctuation, uppercase characters or stop words, and the most frequent tokens (`fix`, `test`, `deploy`, component names…) are exactly the informative words a task classifier needs. The text is ready for feature extraction.

## 6. Save the Processed Dataset

In [9]:
out_cols = ["task_id", "task_description", "description_processed", "category",
            "priority", "status", "created_date", "due_date", "days_to_deadline",
            "estimated_hours", "story_points", "assignee_id",
            "assignee_experience_years", "assignee_open_tasks"]

df[out_cols].to_csv("../data/processed/tasks_nlp.csv", index=False)
print("Saved processed dataset to data/processed/tasks_nlp.csv")
print("Shape:", df[out_cols].shape)

Saved processed dataset to data/processed/tasks_nlp.csv
Shape: (8000, 14)


## 7. Summary of Preprocessing Steps Performed

| Step | What was done | Why |
|---|---|---|
| Lowercasing | All description text lowercased | Removes case variants (`URGENT` vs `urgent`) |
| Punctuation removal | Non-alphanumeric characters replaced by spaces | Strips `!`, `:`, `[ ]`, `-` markers and noise |
| Tokenization | NLTK `word_tokenize` | Splits descriptions into word units |
| Stop-word removal | NLTK English stop-word list (~35% of tokens removed) | Keeps only content-bearing words |
| Lemmatization | WordNet lemmatizer (chosen over Porter stemming) | Normalizes inflections to dictionary words |
| Verification | Vocabulary audit | Confirms text is clean for Week 2 |

**Deliverable:** `data/processed/tasks_nlp.csv` — cleaned dataset plus `description_processed`, ready for TF-IDF / Word2Vec feature extraction and Naive Bayes / SVM classification in Week 2.